In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests


In [2]:
llm = ChatOllama(
    model = "llama3.2:3b",
    temperature=0.4
)

In [3]:
@tool
def get_weather_by_city(city: str) -> dict:
    """Given a city name, return the current weather information including temperature, windspeed, and weather code."""
    try:
        # Step 1: Convert city -> lat, lon using Open-Meteo Geocoding
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
        geo_res = requests.get(geo_url).json()

        results = geo_res.get("results")
        if not results:
            return {"error": f"City '{city}' not found"}

        lat = results[0]["latitude"]
        lon = results[0]["longitude"]
        resolved_city = results[0]["name"]

        # Step 2: Get weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_res = requests.get(weather_url).json()

        weather = weather_res.get("current_weather", {})

        return {
            "city": resolved_city,
            "temperature": weather.get("temperature"),
            "windspeed": weather.get("windspeed"),
            "weathercode": weather.get("weathercode")
        }

    except Exception as e:
        return {"error": str(e)}


In [4]:
print(get_weather_by_city.invoke("Ahmedabad"))

{'city': 'Ahmedabad', 'temperature': 38.2, 'windspeed': 10.2, 'weathercode': 0}


In [5]:
print(get_weather_by_city.args)
print(get_weather_by_city.name)
print(get_weather_by_city.description)

{'city': {'title': 'City', 'type': 'string'}}
get_weather_by_city
Given a city name, return the current weather information including temperature, windspeed, and weather code.


In [6]:
# tool binding

llm_with_tools = llm.bind_tools([get_weather_by_city])

In [7]:
query = HumanMessage(content="What's the current weather in Ahmedabad?")
messages = [query]

In [ ]:
result = llm_with_tools.invoke(messages)

In [9]:
result

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-04-19T07:20:50.337661438Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9199821050, 'load_duration': 5023121175, 'prompt_eval_count': 167, 'prompt_eval_duration': 1835427052, 'eval_count': 21, 'eval_duration': 2270236313, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019da49c-edad-7983-a033-fad952f6403b-0', tool_calls=[{'name': 'get_weather_by_city', 'args': {'city': 'Ahmedabad'}, 'id': '58c5fe8b-a7ac-4c31-bda0-f042395c8e39', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 167, 'output_tokens': 21, 'total_tokens': 188})

In [10]:
messages.append(result)

In [11]:
tool_result = get_weather_by_city.invoke(result.tool_calls[0])

messages.append(tool_result)

In [12]:
messages

[HumanMessage(content="What's the current weather in Ahmedabad?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-04-19T07:20:50.337661438Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9199821050, 'load_duration': 5023121175, 'prompt_eval_count': 167, 'prompt_eval_duration': 1835427052, 'eval_count': 21, 'eval_duration': 2270236313, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019da49c-edad-7983-a033-fad952f6403b-0', tool_calls=[{'name': 'get_weather_by_city', 'args': {'city': 'Ahmedabad'}, 'id': '58c5fe8b-a7ac-4c31-bda0-f042395c8e39', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 167, 'output_tokens': 21, 'total_tokens': 188}),
 ToolMessage(content='{"city": "Ahmedabad", "temperature": 38.2, "windspeed": 10.2, "weathercode": 0}', name='get_weather_by_city', tool_call_id='58c5fe8b-a7ac-4c31-bda0-f

In [13]:
response = llm_with_tools.invoke(messages)

print(response.content)

The current weather in Ahmedabad is mostly sunny with a temperature of 38.2°C and a gentle breeze at 10.2 km/h.


In [14]:
response = llm.invoke("Can you give me that 1$ equals how many INR?")

In [15]:
print(response.content)

The current exchange rate between the US Dollar (USD) and the Indian Rupee (INR) can fluctuate frequently. As of my knowledge cutoff in December 2023, I can provide you with an approximate exchange rate.

1 USD is approximately equal to 79 INR. However, please note that this rate may have changed since my knowledge cutoff date, and I recommend checking a reliable currency conversion source for the most up-to-date information.
